# Regrid country masks from 0.1° x 0.1° to the ERA5 0.25° x 0.25° grid

The 0.1° grid has cell edges at multiples of 0.1°. ERA5 cells are centred on multiples of 0.25°, so their edges sit at 0.125° + n x 0.25°. That is 2.5 x 0.1° per cell, so ERA5 cells cut through 0.1° cells and the integer summation used for the population regrid cannot be used here.


Instead each ERA5 cell gets the **fraction of its area covered by each country**, computed from exact area overlaps (equal-area weighting in latitude, so the polar half-cells and the date-line cell centred on -180° are handled correctly).

Outputs keep the notebook conventions: `lat` south-to-north (-90...90, 721 points), `lon` -180...179.75 (1440 points).

1. `(country, lat, lon)` mask -> float32 area fractions in [0, 1]
2. `(lat, lon)` country-ID map -> plurality country per cell, plus the share of the cell that country covers (`country_id_frac`)

In [1]:
import os
import numpy as np
import xarray as xr
import scipy.sparse as sp
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

mask = xr.open_dataarray(os.path.join(MASKS_DIR, "GBD_Country_Masks_0.10.nc"))
id_mask = xr.open_dataarray(os.path.join(MASKS_DIR, "country_id_mask.nc"))

### Regridding helpers

In [3]:
# ERA5 grid, in this project's ordering (lat S->N, lon -180..179.75). Identical to the population regrid output.
ERA5_LAT = -90 + 0.25 * np.arange(721)
ERA5_LON = -180 + 0.25 * np.arange(1440)


def check_input_grid(da):
    """The weights assume cell centres at -89.95 + 0.1*i and -179.95 + 0.1*j. Fail loudly if not."""
    assert da.sizes["lat"] == 1800 and da.sizes["lon"] == 3600
    assert np.abs(da.lat.values - (-89.95 + 0.1 * np.arange(1800))).max() < 1e-6
    assert np.abs(da.lon.values - (-179.95 + 0.1 * np.arange(3600))).max() < 1e-6


def _overlap_lat(n_fine=1800, res_fine=0.1, n_out=721, res_out=0.25):
    """(721, 1800): share of each ERA5 latitude band lying in each 0.1° band, weighted by
    sin(lat) so shares are area shares. The ±90° bands are half-height. Rows sum to 1."""
    fine_edges = np.deg2rad(-90 + res_fine * np.arange(n_fine + 1))
    centres = -90 + res_out * np.arange(n_out)
    lo = np.deg2rad(np.clip(centres - res_out / 2, -90, 90))
    hi = np.deg2rad(np.clip(centres + res_out / 2, -90, 90))
    W = np.zeros((n_out, n_fine))
    for k in range(n_out):
        a = np.clip(fine_edges[:-1], lo[k], hi[k])
        b = np.clip(fine_edges[1:], lo[k], hi[k])
        w = np.clip(np.sin(b) - np.sin(a), 0, None)
        W[k] = w / w.sum()
    return sp.csr_matrix(W)


def _overlap_lon(n_fine=3600, res_fine=0.1, n_out=1440, res_out=0.25):
    """(1440, 3600): same in longitude. The cell centred on -180° spans -180.125..-179.875, so its
    western half wraps to +179.875..180; the ±360 copies fold the two ends of the grid together."""
    fine_edges = -180 + res_fine * np.arange(n_fine + 1)
    centres = -180 + res_out * np.arange(n_out)
    W = np.zeros((n_out, n_fine))
    for l in range(n_out):
        lo, hi = centres[l] - res_out / 2, centres[l] + res_out / 2
        for shift in (-360, 0, 360):
            a = np.clip(fine_edges[:-1] + shift, lo, hi)
            b = np.clip(fine_edges[1:] + shift, lo, hi)
            W[l] += np.clip(b - a, 0, None)
        W[l] /= W[l].sum()
    return sp.csr_matrix(W)


W_LAT, W_LON = _overlap_lat(), _overlap_lon()


def regrid_fraction(indicator_2d):
    """(1800, 3600) 0/1 indicator -> (721, 1440) float32 area fraction in [0, 1]."""
    x = np.asarray(indicator_2d, dtype=np.float32)
    return np.asarray(W_LAT @ (W_LON @ x.T).T, dtype=np.float32)

### 1. `(country, lat, lon)` mask -> area fractions

A value of 0.37 means 37% of that ERA5 cell's area lies inside the country. For country aggregates use these as weights, e.g. `(pop * frac).sum(...)`, rather than thresholding them. Thresholding drops small countries/enclaves and mis-assigns border cells.

In [4]:
check_input_grid(mask)
assert mask.dims == ("country", "lat", "lon"), mask.dims

n_country = mask.sizes["country"]
frac = np.zeros((n_country, 721, 1440), dtype=np.float32)
for i in range(n_country):                       # one country at a time keeps memory low
    frac[i] = regrid_fraction(mask.isel(country=i).values)

# carry over every non-spatial coordinate (country names, region, ...) unchanged
coords = {k: v for k, v in mask.coords.items() if k not in ("lat", "lon")}
coords.update(lat=ERA5_LAT, lon=ERA5_LON)

mask_era5 = xr.DataArray(
    frac, dims=("country", "lat", "lon"), coords=coords,
    name=mask.name or "country_mask",
    attrs={
        "description": ("Fraction (0-1) of each 0.25°x0.25° ERA5-centred cell's area covered by each "
                        "country, area-weighted from the 0.1°x0.1° binary masks."),
        "citation": mask.attrs.get("citation", ""),
        "grid": "ERA5 0.25° (cells centred on multiples of 0.25°), lat -90..90, lon -180..179.75",
    },
)

out_path = os.path.join(MASKS_DIR, "GBD_Country_Masks_0.25.nc")
print(f"Saving to {out_path}")
mask_era5.to_netcdf(out_path, encoding={mask_era5.name: {"zlib": True, "complevel": 4}})

Saving to /glade/work/awells/workflow/BMR/masks/country/GBD_Country_Masks_0.25.nc


### 2. `(lat, lon)` country-ID map

A single ID per cell cannot represent a border cell, so each cell gets the **country covering the largest share of it** (ties go to the smaller ID). `country_id_frac` stores that share so you can filter later (e.g. `>= 0.5`). Cells touched by no country stay `NaN`.

Built from the existing ID map, so the 53 original overlap cells keep the "first match" resolution already applied there.

In [5]:
check_input_grid(id_mask)
v = id_mask.values
ids = np.unique(v[~np.isnan(v)]).astype(int)  # ascending, so strict '>' below gives ties to the smaller ID

best_frac = np.zeros((721, 1440), dtype=np.float32)
best_id = np.full((721, 1440), np.nan)
total = np.zeros((721, 1440), dtype=np.float32)  # share of cell covered by any country
for cid in ids:
    f = regrid_fraction(v == cid)
    better = f > best_frac
    best_id[better] = cid
    best_frac[better] = f[better]
    total += f

unlabelled = best_frac <= 1e-6  # real overlaps are >= 0.01 of a cell, so this only removes float zeros
best_id[unlabelled] = np.nan
best_frac = np.where(unlabelled, np.nan, best_frac).astype(np.float32)

id_era5 = xr.Dataset(
    {
        "country_id": (("lat", "lon"), best_id,
                       {"description": ("Country ID per 0.25° ERA5-centred cell: the country covering the largest "
                                        "share of the cell's area. Derived from the 0.1° country_id mask."),
                        "id_source": id_mask.attrs.get("id_source", "country_location_ids.json"),
                        "nodata_value": "NaN (no country covers any part of the cell)"}),
        "country_id_frac": (("lat", "lon"), best_frac,
                            {"description": "Share (0-1) of the cell's area covered by country_id's country."}),
    },
    coords={"lat": ERA5_LAT, "lon": ERA5_LON},
)

out_path = os.path.join(MASKS_DIR, "country_id_mask_0.25.nc")
print(f"Saving to {out_path}")
id_era5.to_netcdf(out_path)

Saving to /glade/work/awells/workflow/BMR/masks/country/country_id_mask_0.25.nc


### Checks

In [6]:
# 1. Total country coverage of a cell can never exceed the cell
assert total.max() <= 1 + 1e-5, total.max()

# 2. Same grid as the ERA5 / population products
assert np.array_equal(id_era5.lat.values, ERA5_LAT) and np.array_equal(id_era5.lon.values, ERA5_LON)
# (and, with an ERA5 file converted to lat S->N, lon -180..179.75:  np.array_equal(era5_conv.lat.values, ERA5_LAT) etc.)

# 3. Soft mask vs ID map coverage. They should agree except where the ID map has no ID for a country
#    (names missing from the JSON) or where the original 53 overlap cells were resolved by "first match".
diff = np.abs(frac.sum(axis=0) - total)
print(f"cells where soft-mask coverage and ID-map coverage differ by > 1e-5: {(diff > 1e-5).sum()} (max {diff.max():.3f})")

# 4. Summary
lab = ~np.isnan(best_id)
print(f"labelled cells: {lab.sum()} ({lab.mean():.2%} of grid)")
print(f"plurality share == 1: {(best_frac[lab] > 0.9999).mean():.1%}, >= 0.5: {(best_frac[lab] >= 0.5).mean():.1%}")
lost = sorted(set(ids) - set(np.unique(best_id[lab]).astype(int)))
print("IDs that win no cell at 0.25° (small countries/enclaves; present in the soft mask only):", lost)

cells where soft-mask coverage and ID-map coverage differ by > 1e-5: 93 (max 0.320)
labelled cells: 260320 (25.07% of grid)
plurality share == 1: 85.0%, >= 0.5: 94.3%
IDs that win no cell at 0.25° (small countries/enclaves; present in the soft mask only): [396]


In [22]:
# === Diagnose: where does the soft mask 'see' more than the ID map, and why? ===
from collections import Counter

# Number of country layers that are 1 in each 0.1° cell (loop keeps memory low)
n_layers = np.zeros((1800, 3600), dtype=np.int16)
for i in range(mask.sizes["country"]):
    n_layers += mask.isel(country=i).values.astype(np.int16)

has_id = ~np.isnan(id_mask.values)
overlap_extra = np.where(has_id, np.clip(n_layers - 1, 0, None), 0)   # in >1 layer; the ID map kept only the first
no_id = np.where(~has_id, n_layers, 0)                                 # in a layer, but NaN in the ID map
print(f"0.1° cells in >1 country layer: {(overlap_extra > 0).sum()} | in a layer but NaN in the ID map: {(no_id > 0).sum()}")

0.1° cells in >1 country layer: 53 | in a layer but NaN in the ID map: 0


In [23]:
# By linearity, soft-mask coverage minus ID-map coverage must equal the regrid of (layers - ID-map coverage)
predicted = regrid_fraction((n_layers - has_id).astype(np.float32))
actual = frac.sum(axis=0) - total
print(f"max |actual - predicted|: {np.abs(actual - predicted).max():.2e}   (~1e-6 or less => fully explained)")
print(f"cells where soft mask is LOWER than the ID map (would indicate a bug): {(actual < -1e-5).sum()}")

max |actual - predicted|: 1.01e-07   (~1e-6 or less => fully explained)
cells where soft mask is LOWER than the ID map (would indicate a bug): 0


In [24]:
# The ERA5 cells that differ, largest first
ky, kx = np.nonzero(np.abs(actual) > 1e-5)
print(f"\nERA5 cells differing: {len(ky)} | max {actual.max():.3f} | distribution of differences:",
      np.round(np.percentile(actual[ky, kx], [0, 50, 90, 100]), 3))
for o in np.argsort(-actual[ky, kx])[:8]:
    print(f"  lat {ERA5_LAT[ky[o]]:7.2f}  lon {ERA5_LON[kx[o]]:8.2f}  soft-mask coverage exceeds ID map by {actual[ky[o], kx[o]]:.3f}")



ERA5 cells differing: 93 | max 0.320 | distribution of differences: [0.01 0.09 0.16 0.32]
  lat   34.00  lon    70.50  soft-mask coverage exceeds ID map by 0.320
  lat   33.50  lon    78.75  soft-mask coverage exceeds ID map by 0.280
  lat   29.25  lon    95.75  soft-mask coverage exceeds ID map by 0.210
  lat   42.50  lon    20.00  soft-mask coverage exceeds ID map by 0.200
  lat   29.50  lon    96.00  soft-mask coverage exceeds ID map by 0.170
  lat   27.00  lon    85.00  soft-mask coverage exceeds ID map by 0.170
  lat   45.50  lon    28.25  soft-mask coverage exceeds ID map by 0.160
  lat   31.50  lon    79.00  soft-mask coverage exceeds ID map by 0.160


In [25]:
# Which countries share the overlapping 0.1° cells?
oy, ox = np.nonzero(overlap_extra)
pairs = []
for y, x in zip(oy, ox):
    names = mask["country"].values[mask.isel(lat=y, lon=x).values.astype(bool)]
    pairs.append(tuple(sorted(map(str, names))))
print("\nCountry combinations sharing a 0.1° cell (count):", Counter(pairs).most_common(10))


Country combinations sharing a 0.1° cell (count): [(('China', 'India'), 14), (('Afghanistan', 'Pakistan'), 10), (('Bangladesh', 'India'), 6), (('India', 'Nepal'), 6), (('India', 'Myanmar'), 2), (('Albania', 'Montenegro'), 2), (('Iran', 'Pakistan'), 1), (('Bhutan', 'India'), 1), (('China', 'Pakistan'), 1), (('North Korea', 'Russian Federation'), 1)]


In [26]:
# Countries that can never appear in a country map built from this mask
cells = mask.sum(("lat", "lon")).values
print("countries with an EMPTY 0.1° layer:", list(mask["country"].values[cells == 0]))

countries with an EMPTY 0.1° layer: ['Monaco', 'Marshall Islands', 'Nauru', 'Tokelau', 'Tuvalu']


In [28]:
# Countries that have cells in the mask but ZERO cells in the ID map (lost entirely to "first match", or sharing an ID).
# Needs the name -> ID JSON used in 0b:  JSON_DIR / "country_location_ids.json"
import json
JSON_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
name_to_id = json.load(open(os.path.join(JSON_DIR, "country_location_ids.json")))
in_map = np.array([(id_mask.values == name_to_id.get(str(n), -1)).sum() for n in mask["country"].values])
print("countries with cells in the mask but none in the ID map:",
      [(str(n), int(c)) for n, c, m in zip(mask["country"].values, cells, in_map) if c > 0 and m == 0])
print("names missing from the JSON:", [str(n) for n in mask["country"].values if str(n) not in name_to_id])
ids_per_name = Counter(name_to_id.get(str(n)) for n in mask["country"].values)
print("IDs shared by more than one mask country:", [k for k, c in ids_per_name.items() if c > 1])

countries with cells in the mask but none in the ID map: []
names missing from the JSON: []
IDs shared by more than one mask country: []
